# ADMMIN MODEL

In [1]:
from sqlalchemy.dialects.mysql import base 
base.ischema_names['tinyint'] = base.BOOLEAN
base.ischema_names['mediumtext'] = base.TEXT 
# from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Text,
    Time,
    DateTime,
    Date,
    Float,
    Boolean,
    ForeignKey,
    UniqueConstraint
)
from sqlalchemy.orm import sessionmaker, declarative_base
from sqlalchemy import inspect
from sqlalchemy import MetaData
from sqlalchemy import Table as _Table
from sqlalchemy.sql import select, and_
from sqlalchemy.sql import insert
from sqlalchemy.sql import update
from sqlalchemy.sql import delete
from sqlalchemy.sql import text
#from sqlalchemy.sql.expression import func

from sqlalchemy import func
import datetime
from dateutil import parser
import json
import hashlib
import os
from sqlalchemy.types import TypeDecorator, Unicode
import sys
from sqlalchemy.engine.url import URL
import copy
#from sqlalchemy_utils import database_exists, create_database
import re
import pandas as pd
import numpy as np
import sqlalchemy as sa
import openpyxl as xl
from passlib.hash import pbkdf2_sha256
from passlib.context import CryptContext
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
from sqlalchemy_utils import database_exists, create_database
import uuid

# ADMIN DB

In [2]:
database = 'database/ADMIN'

In [3]:
os.path.split(database)

('database', 'ADMIN')

In [4]:
db_conf = {
    "drivername": "mysql+pymysql",
    "username": "root",
    "password": "1234",
    "host": "localhost",
    "port": 3306,
    "database": database
}
db_conf = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "1234",
    "host": "localhost",
    "port": 5432,
    "database": os.path.split(database)[1] if len(os.path.split(database)) > 1 else database
}
db_conf = {
    "drivername": "sqlite",
    "database": f"{database}.db"
}
url = URL.create(**db_conf)
engine = create_engine(url, echo = False)
if not database_exists(engine.url):
    create_database(engine.url)

## SESSION

In [5]:
Session = sessionmaker(bind = engine, autoflush = True)
session = Session()
Base = declarative_base()
metadata = MetaData()
metadata.reflect(engine)
inspector = inspect(engine)

# APP DB

In [6]:
app_database = 'database/ETLX'

In [7]:
db_conf = {
    "drivername": "mysql+pymysql",
    "username": "root",
    "password": "1234",
    "host": "localhost",
    "port": 3306,
    "database": app_database
}
db_conf = {
    "drivername": "postgresql+psycopg2",
    "username": "postgres",
    "password": "1234",
    "host": "localhost",
    "port": 5432,
    "database": os.path.split(app_database)[1] if len(os.path.split(app_database)) > 1 else app_database
}
db_conf = {
    "drivername": "sqlite",
    "database": f"{app_database}.db"
}
url = URL.create(**db_conf)
app_engine = create_engine(url, echo = False)
try:
    if not database_exists(app_engine.url):
        create_database(app_engine.url)
except Exception as e:
    print(str(e))

In [8]:
url

sqlite:///database/ETLX.db

In [9]:
app_database = os.path.split(app_database)[1]
app_database

'ETLX'

## SESSION

In [10]:
Session = sessionmaker(bind = app_engine, autoflush = True)
session_app = Session()
Base = declarative_base()
metadata_app = MetaData()
metadata_app.reflect(app_engine)
inspector_app = inspect(app_engine)

In [11]:
now = datetime.datetime.now()
_data = {}

# ETLX

In [12]:
class ETLX(Base):
    __tablename__ = 'etlx'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine' : 'InnoDB',
        'comment'      : 'ETLX',
        'mysql_charset': 'utf8',
        'sqlite_autoincrement': True
    }
    etlx_id           = Column(Integer, primary_key = True, autoincrement = True, comment = "ID")
    etl               = Column(String(200), unique = True, nullable = False, comment = "Name")
    etl_desc          = Column(Text, nullable = True, comment = "Description")
    attach_etlx_conf  = Column(String(200), nullable = True, comment = "Config File")
    etlx_conf         = Column(Text, nullable = True, comment = "Config Text")
    active            = Column(Boolean, default = True, comment = "Active")
    user_id           = Column(Integer, comment = "User ID")
    app_id            = Column(Integer, comment = "App ID")
    created_at        = Column(DateTime, nullable = True, comment = "Created at")
    updated_at        = Column(DateTime, nullable = True, comment = "Updated at")
    excluded          = Column(Boolean, default = False, comment = "Excluded")

In [13]:
class ETLXConf(Base):
    __tablename__ = 'etlx_conf'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine' : 'InnoDB',
        'comment'      : 'ETLX Extra Cofig',
        'mysql_charset': 'utf8',
        'sqlite_autoincrement': True
    }
    etlx_conf_id      = Column(Integer, primary_key = True, autoincrement = True, comment = "ID")
    etlx_conf         = Column(String(200), unique = True, nullable = False, comment = "Name")
    etlx_conf_desc    = Column(Text, nullable = True, comment = "Description")
    etlx_extra_conf   = Column(Text, nullable = True, comment = "Config Text")
    user_id           = Column(Integer, comment = "User ID")
    app_id            = Column(Integer, comment = "App ID")
    created_at        = Column(DateTime, nullable = True, comment = "Created at")
    updated_at        = Column(DateTime, nullable = True, comment = "Updated at")
    excluded          = Column(Boolean, default = False, comment = "Excluded")

## QUERY MANEGMENT

In [14]:
class ManageQuery(Base):
    __tablename__ = 'manage_query'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine' : 'InnoDB',
        'comment'      : 'Queries',
        'mysql_charset': 'utf8',
        'sqlite_autoincrement': True
    }
    manage_query_id    = Column(Integer, primary_key = True, autoincrement = True, comment = "ID")
    manage_query       = Column(String(200), unique = False, nullable = False, comment = "Query Desc")
    database           = Column(String(200), nullable = False, comment = "Database") 
    manage_query_conf  = Column(Text, nullable = True, comment = "Query Config")
    active             = Column(Boolean, default = True, comment = "Active")
    user_id            = Column(Integer,  comment = "User ID")
    app_id             = Column(Integer, comment = "App ID")
    created_at         = Column(DateTime, nullable = True, comment = "Created at")
    updated_at         = Column(DateTime, nullable = True, comment = "Updated at")
    excluded           = Column(Boolean, default = False, comment = "Excluded")

# DASHBOARD

In [15]:
class Dashboard(Base):
    __tablename__ = 'dashboard'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine'   : 'InnoDB',
        'comment'        : 'Dashboards',
        'mysql_charset'  : 'utf8',
        'sqlite_autoincrement': True
    }
    dashboard_id   = Column(Integer, primary_key = True, autoincrement = True, comment = "Dashboard ID")
    dashboard      = Column(String(200), comment = "Dashboard")
    dashboard_desc = Column(Text, nullable = True, comment = "Description")
    dashboard_conf = Column(Text, nullable = False, comment = "Conf / Params")
    order          = Column(Integer, nullable = True, comment = "Order")
    active         = Column(Boolean, default = True, comment = "Active")
    user_id        = Column(Integer,  comment = "User ID")
    app_id         = Column(Integer, comment = "App ID")
    created_at     = Column(DateTime, nullable = True, comment = "Created at")
    updated_at     = Column(DateTime, nullable = True, comment = "Updated at")
    excluded       = Column(Boolean, default = False, comment = "Excluded")

## DASHBOARD COMMENTS

Make a section an the dashboard whre the user can add comments

In [16]:
class DashboardComment(Base):
    __tablename__ = 'dashboard_comment'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine'   : 'InnoDB',
        'comment'        : 'Dashboards Comments',
        'mysql_charset'  : 'utf8',
        'sqlite_autoincrement': True
    }
    dashboard_comment_id = Column(Integer, primary_key = True, autoincrement = True, comment = "Comment ID")
    dashboard_comment    = Column(Text, comment = "Comments")
    dashboard            = Column(String(200), nullable = True, comment = "Dashboard")
    active               = Column(Boolean, default = True, comment = "Active")
    user_id              = Column(Integer,  comment = "User ID")
    app_id               = Column(Integer, comment = "App ID")
    created_at           = Column(DateTime, nullable = True, comment = "Created at")
    updated_at           = Column(DateTime, nullable = True, comment = "Updated at")
    excluded             = Column(Boolean, default = False, comment = "Excluded")

# NOTEBOOK

In [17]:
class Notebook(Base):
    __tablename__ = 'notebook'
    __table_args__ = {
        'extend_existing': True,
        'mysql_engine'   : 'InnoDB',
        'comment'        : 'Notebooks',
        'mysql_charset'  : 'utf8',
        'sqlite_autoincrement': True
    }
    notebook_id   = Column(Integer, primary_key = True, autoincrement = True, comment = "Notebook ID")
    notebook      = Column(String(200), comment = "Name")
    notebook_desc = Column(Text, nullable = True, comment = "Description")
    notebook_conf = Column(Text, nullable = False, comment = "Conf / Params")
    active        = Column(Boolean, default = True, comment = "Active")
    user_id       = Column(Integer,  comment = "User ID")
    app_id        = Column(Integer, comment = "App ID")
    created_at    = Column(DateTime, nullable = True, comment = "Created at")
    updated_at    = Column(DateTime, nullable = True, comment = "Updated at")
    excluded      = Column(Boolean, default = False, comment = "Excluded")

# DROP FIRST

In [18]:
#engine.dispose()

In [19]:
#TaskTrack.__table__.drop(bind = app_engine, checkfirst = True)

# DROP ALL FIRST

In [20]:
#session.close()
try:
    pass
    #Base.metadata.drop_all(bind = app_engine, checkfirst = True)
except Exception as e:
    print(str(e))

# CREATE ALL

In [21]:
Base.metadata.create_all(bind = app_engine, checkfirst = True)

In [22]:
app_engine.url

sqlite:///database/ETLX.db

# ADD DATA

In [23]:
#session.close()
tables = []
for cls in Base.__subclasses__():
    tables.append(cls.__tablename__)
print(tables)

['etlx', 'etlx_conf', 'manage_query', 'dashboard', 'dashboard_comment', 'notebook']


In [24]:
_updated_tables = ['notebook']
tables = []
for cls in Base.__subclasses__():
    tables.append(cls.__tablename__)
print(tables)

['etlx', 'etlx_conf', 'manage_query', 'dashboard', 'dashboard_comment', 'notebook']


In [25]:
_data.keys()

dict_keys([])

In [26]:
app_engine.url

sqlite:///database/ETLX.db

In [27]:
for table in _data:
    if not _updated_tables:
        pass
    elif len(_updated_tables) == 0:
        pass
    elif table not in _updated_tables:
        continue    
    print(table, '...')
    tbl = _Table(table, metadata_app, autoload_with = app_engine)
    with app_engine.connect() as conn_admin:
        try: 
            sql = insert(tbl).values(_data.get(table))
            result = conn_admin.execute(sql)
            print(table, result.rowcount, 'rows inserted')
            result.close()
            conn_admin.commit()
            conn_admin.close()
        except Exception as e:
            print(str(e)[0:100])

In [28]:
tables = Base.metadata.tables
print(tables)

FacadeDict({'etlx': Table('etlx', MetaData(), Column('etlx_id', Integer(), table=<etlx>, primary_key=True, nullable=False, comment='ID'), Column('etl', String(length=200), table=<etlx>, nullable=False, comment='Name'), Column('etl_desc', Text(), table=<etlx>, comment='Description'), Column('attach_etlx_conf', String(length=200), table=<etlx>, comment='Config File'), Column('etlx_conf', Text(), table=<etlx>, comment='Config Text'), Column('active', Boolean(), table=<etlx>, default=ScalarElementColumnDefault(True), comment='Active'), Column('user_id', Integer(), table=<etlx>, comment='User ID'), Column('app_id', Integer(), table=<etlx>, comment='App ID'), Column('created_at', DateTime(), table=<etlx>, comment='Created at'), Column('updated_at', DateTime(), table=<etlx>, comment='Updated at'), Column('excluded', Boolean(), table=<etlx>, default=ScalarElementColumnDefault(False), comment='Excluded'), schema=None), 'etlx_conf': Table('etlx_conf', MetaData(), Column('etlx_conf_id', Integer()

# REGISTER ALL THE TABLE AND THE COMMENTS / LABELS

In [29]:
i = 1
j = 1
now = datetime.datetime.now()
_data = {}
for cls in Base.__subclasses__():
    if not _updated_tables:
        pass
    elif len(_updated_tables) == 0:
        pass
    elif cls.__tablename__ not in _updated_tables:
        continue
    args = cls.__table_args__
    if not _data.get('table'):
        _data['table'] = []
    _data['table'].append({
        'table': cls.__tablename__,
        'table_desc': args['comment'],
        'db': app_database,
        'user_id': 1,
        'created_at': now,
        'updated_at': now,
        'excluded': False
    })
    if not _data.get('translate_table'):
        _data['translate_table'] = []
    _data['translate_table'].append({
        'table_org_desc': args['comment'],
        'table_transl_desc': args['comment'],
        'table': cls.__tablename__,
        'db': app_database,
        'lang': 'en',
        'user_id': 1,
        'created_at': now,
        'updated_at': now,
        'excluded': False
    })
    for c in cls.__table__.columns:
        if not _data.get('translate_table_field'):
            _data['translate_table_field'] = []
        _data['translate_table_field'].append({
                'field_org_desc': c.comment,
                'field_transl_desc': c.comment,
                'field': str(c.name),
                'table': cls.__tablename__,
                'db': app_database,
                'lang': 'en',
                'user_id': 1,
                'created_at': now,
                'updated_at': now,
                'excluded': False
            })
        if not _data.get('table_schema'):
            _data['table_schema'] = []
        fk = False
        referred_table = None
        referred_column = None
        if c.foreign_keys:
            fk = True
            for _fk in c.foreign_keys:
                #print(f"  Foreign Key to: {_fk.column.table.name}.{_fk.column.name}")
                #print(f"  On Table: {_fk.column.table}")
                referred_table = str(_fk.column.table)
                #print(f"  Referenced Column: {_fk.column}")
                referred_column = str(_fk.column.name).replace(f'{referred_table}.', '')
            #print(fk, referred_table, referred_column)
        default_value = None
        if c.default is not None:
            default_value = c.default.arg
            #print(c.name, default_value)
        _data['table_schema'].append({
                'table': cls.__tablename__,
                'db': app_database,
                'field': str(c.name),
                'pk': c.primary_key,
                'type': str(c.type),
                'nullable': c.nullable,
                'default': default_value,
                'autoincrement': None if c.autoincrement == 'auto' else c.autoincrement,
                'comment': str(c.comment),
                'computed': c.computed,
                'fk': fk,
                'referred_table': referred_table,
                'referred_column': referred_column,
                'user_id': 1,
                'created_at': now,
                'updated_at': now,
                'excluded': False
            })
        j += 1
    i += 1

In [30]:
chk_first = True
for table in _data:
    print(table, '...')
    tbl = _Table(table, metadata, autoload_with = engine)
    sql = delete(tbl).where(tbl.c.db == app_database)
    with engine.connect() as conn_admin:
        if chk_first == True:
            for d in _data.get(table):
                for k in d:
                    if not d.get(k):
                        pass
                    elif str(d.get(k, '')).find('-01:00') != -1:
                        print(k, d[k])
                        d[k] = parser.parse((d[k]).replace('-01:00', ''))
                sql = select(tbl.c)\
                    .select_from(tbl)\
                    .where(and_(
                        tbl.c.db == app_database,
                        tbl.c.table == d.get('table')
                    ))
                if table in ['translate_table_field', 'table_schema']:
                    sql = select(tbl.c)\
                        .select_from(tbl)\
                        .where(and_(
                            tbl.c.db == app_database,
                            tbl.c.table == d.get('table'),
                            tbl.c.field == d.get('field')
                        ))
                _curr = conn_admin.execute(sql).fetchone()
                if _curr:
                    # print('_curr:', _curr)
                    sql = update(tbl)\
                        .where(and_(
                            tbl.c.db == app_database,
                            tbl.c.table == d.get('table')
                        ))\
                        .values(d)
                    if table in ['translate_table_field', 'table_schema']:
                        sql = update(tbl)\
                            .where(and_(
                                tbl.c.db == app_database,
                                tbl.c.table == d.get('table'),
                                tbl.c.field == d.get('field')
                            )).values(d)
                    result = conn_admin.execute(sql)
                    print(d.get('table'), d.get('field'), result.rowcount, 'rows updated')
                    result.close()
                else:
                    sql = insert(tbl).values(d)
                    result = conn_admin.execute(sql)
                    print(table, result.rowcount, 'rows inserted')
                    result.close()
        else:
            result = conn_admin.execute(sql)
            print(table, result.rowcount, 'rows deleted')
            result.close()
            sql = insert(tbl).values(_data.get(table))
            result = conn_admin.execute(sql)
            print(table, result.rowcount, 'rows inserted')
            result.close()
        conn_admin.commit()
        conn_admin.close()

table ...
table 1 rows inserted
translate_table ...
translate_table 1 rows inserted
translate_table_field ...
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
translate_table_field 1 rows inserted
table_schema ...
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted
table_schema 1 rows inserted


# COMMIT

In [32]:
commit = True
#session.rollback()

In [33]:
try:
    if commit == True:
        session.commit()
except Exception as e:
    print(str(e))

# CONFIG INTERFACE

In [31]:
dash_conf = {
    "label": "dashboard",
    "tooltip": "dashboard_desc",
    "load_items": {
        "table": "dashboard",
        "tables": ["dashboard"]
    }
}
notebook_conf = {
    "label": "notebook",
    "tooltip": "notebook_desc",
    "load_items": {
        "table": "notebook",
        "tables": ["notebook"]
    }
}


In [32]:
interface_conf = {
    'Dashboards': {
        'menu_icon': 'document-report',
        'menu_order': 1,
        'active': True,
        'menu_config': json.dumps(dash_conf),
        'tables': [
            'dashboard'
        ]
    },
    'ETLX':{
        'menu_icon': 'circle-stack',
        'menu_order': 2,
        'active': True,
        #'menu_config': json.dumps(etl_conf),
        'tables': [
            'etlx',
            'etlx_conf',
            'manage_query',
        ]
    },
    'Notebook': {        
        'menu_icon': 'book-open',
        'menu_order': 3,
        'active': True,
        'menu_config': json.dumps(notebook_conf),
        'tables': [
            'notebook',
        ]
    }
}

In [33]:
tbl_app = _Table('app', metadata, autoload_with = engine)
sql = select(tbl_app.c)\
    .select_from(tbl_app)\
    .where(tbl_app.c.excluded == True)
with engine.connect() as conn_admin:
    apps = conn_admin.execute(sql).fetchall()
    print(len(apps), apps)

0 []


In [34]:
def _run(interface_conf):
    tbl_app = _Table('app', metadata, autoload_with = engine)
    tbl_menu = _Table('menu', metadata, autoload_with = engine)
    tbl_table = _Table('table', metadata, autoload_with = engine)
    tbl_menu_table = _Table('menu_table', metadata, autoload_with = engine)
    sql = select(tbl_app.c)\
        .select_from(tbl_app)\
        .where(tbl_app.c.db == app_database)
    now = datetime.datetime.now()
    with engine.connect() as conn_admin:
        app = conn_admin.execute(sql).fetchone()
        if not app:
            print('THE APP DOES NOT EXISTS!')
            data = {
                'app': app_database,
                'app_desc': 'ETLX UI in CS',
                'version': '1.0.0',
                'db': app_database,
                'user_id': 1,
                'created_at': now,
                'updated_at': now,
                'excluded': False
            }
            try:
                sql2 = insert(tbl_app).values(data)
                result = conn_admin.execute(sql2)
                result.close()
                app = conn_admin.execute(sql).fetchone()
            except Exception as _err:
                #print(str(_err))
                if re.match(r'UniqueViolation', str(_err)) and re.match(r'postgres', str(engine.url)):
                    _sql = """SELECT setval(pg_get_serial_sequence('"{table}"', '{field}'), coalesce(max("{field}") + 1, 1), false) 
                            FROM "{table}";""".format(table = 'app', field = 'app_id')
                    _sql = """SELECT SETVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}'), NEXTVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}')), FALSE)""".format(table = 'app', field = 'app_id')
                    conn_admin.execute(_sql)
                    result = conn_admin.execute(sql2)
                    result.close()                
        for m in interface_conf:
            if interface_conf[m].get('active') == True:
                sql = select(tbl_menu.c)\
                        .select_from(tbl_menu)\
                        .where(and_(
                            tbl_menu.c.menu == m,
                            tbl_menu.c.app_id == app.app_id
                        ))
                menu = conn_admin.execute(sql).fetchone()
                print(menu)
                if menu:
                    print(m, menu.menu_id)
                else:
                    data = {
                        'menu':  m,
                        'menu_desc': m,
                        'menu_icon': interface_conf[m].get('menu_icon'),
                        'active': interface_conf[m].get('active'),
                        'menu_order': interface_conf[m].get('menu_order'),
                        'menu_config': interface_conf[m].get('menu_config'),
                        'app_id': app.app_id,
                        'user_id': 1,
                        'created_at': now,
                        'updated_at': now,
                        'excluded': False
                    }
                    try:
                        sql2 = insert(tbl_menu).values(data)
                        result = conn_admin.execute(sql2)
                        result.close()
                        menu = conn_admin.execute(sql).fetchone()
                    except Exception as _err:
                        #print(str(_err))
                        if re.match(r'UniqueViolation', str(_err)) and re.match(r'postgres', str(engine.url)):
                            _sql = """SELECT SETVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}'), NEXTVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}')), FALSE)""".format(table = 'menu', field = 'menu_id')
                            conn_admin.execute(_sql)
                            result = conn_admin.execute(sql2)
                            result.close() 
                    print(m, 'Adicionado: ', menu.menu_id)
                # MENU TABELA
                if interface_conf[m].get('tables'):
                    for t in interface_conf[m].get('tables'):
                        _active = True
                        if type(t) == dict:
                            _tbl = t.get('table')
                            _active = t.get('active')
                        else:
                            _tbl = t
                        _requires_rla = False
                        if type(t) == dict:
                            _requires_rla = t.get('requires_rla')
                        sql = select(tbl_table.c)\
                            .select_from(tbl_table)\
                            .where(and_(
                                tbl_table.c.table == _tbl,
                                tbl_table.c.db == app_database
                            ))
                        table = conn_admin.execute(sql).fetchone()
                        # print(_tbl, app_database, table)
                        try:
                            # print(_tbl, app_database, table, table)
                            sql = select(tbl_menu_table.c)\
                                    .select_from(tbl_menu_table)\
                                    .where(and_(
                                        tbl_menu_table.c.table_id == table.table_id,
                                        tbl_menu_table.c.menu_id == menu.menu_id,
                                        tbl_menu_table.c.app_id == app.app_id
                                    ))
                            menu_table = conn_admin.execute(sql).fetchone()
                            if not menu_table:
                                data = {
                                    'menu_id': menu.menu_id,
                                    'table_id': table.table_id,
                                    'app_id': app.app_id,
                                    'active': _active,
                                    'requires_rla': _requires_rla,
                                    'user_id': 1,
                                    'created_at': now,
                                    'updated_at': now,
                                    'excluded': False
                                }
                                try:
                                    sql2 = insert(tbl_menu_table).values(data)
                                    result = conn_admin.execute(sql2)
                                    result.close()
                                    menu_table = conn_admin.execute(sql).fetchone()
                                except Exception as _err:
                                    #print(str(_err))
                                    if re.match(r'UniqueViolation', str(_err)) and re.match(r'postgres', str(engine.url)):
                                        _sql = """SELECT SETVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}'), NEXTVAL(PG_GET_SERIAL_SEQUENCE('{table}', '{field}')), FALSE)""".format(table = 'menu_table', field = 'menu_table_id')
                                        conn_admin.execute(_sql)
                                        result = conn_admin.execute(sql2)
                                        result.close() 
                        except Exception as e:
                            print(_tbl, table, str(e))
        conn_admin.commit()

In [35]:
_run(interface_conf)

(7, 'Dashboards', 'Dashboards', 'document-report', 1, '{"label": "dashboard", "tooltip": "dashboard_desc", "load_items": {"table": "dashboard", "tables": ["dashboard"]}}', 3, 1, True, datetime.datetime(2025, 3, 5, 15, 29, 12, 313089), datetime.datetime(2025, 3, 5, 15, 29, 12, 313089), False)
Dashboards 7
(8, 'ETLX', 'ETLX', 'circle-stack', 2, None, 3, 1, True, datetime.datetime(2025, 3, 5, 15, 29, 12, 313089), datetime.datetime(2025, 3, 5, 15, 29, 12, 313089), False)
ETLX 8
manage_query None 'NoneType' object has no attribute 'table_id'
None
Notebook Adicionado:  12


In [70]:
from datetime import datetime

iso_string = '2024-10-19 17:47:09.5611122'
fixed_iso_string = iso_string[:26]  # Trim to 6 microseconds
dt = datetime.fromisoformat(fixed_iso_string)

print(dt)


2024-10-19 17:47:09.561112


# SYNC CURRENT DB TABLES / FROMS CONF

uses the developer configuratio / contumization of the tables and views to putit avaliable for future instalations in diferent machines

In [41]:
tables = Base.metadata.tables

In [42]:
_tables_conf = ['custom_table', 'custom_form']
with engine.connect() as conn:
    for _table in _tables_conf:
        _sql = f"""SELECT * FROM {_table} WHERE db = '{app_database}' AND excluded IS FALSE"""
        _df = pd.read_sql(_sql, con = conn)
        if _df.shape[0] > 0:
            _df.to_json(f'{_table}.{app_database}.json', orient = 'records', force_ascii = False)
            print(_df.shape)

(3, 9)
(3, 9)


# USE SYNCED TABLE / FROMS CONF

In [121]:
_tables_conf = ['custom_table', 'custom_form']
_tbls = {}
with engine.connect() as conn:
    tbl_app = _Table('app', metadata, autoload_with = engine)
    sql = select(tbl_app.c)\
        .select_from(tbl_app)\
        .where(tbl_app.c.db == app_database)
    app = conn.execute(sql).fetchone()
    for _table in _tables_conf:
        try:
            if os.path.exists(f'{_table}.{app_database}.json'):
                _df = pd.read_json(f'{_table}.{app_database}.json')
            else:
                _df = pd.read_json(f'{_table}.json')
            #print(_df.shape[0])
            if _df.shape[0] > 0:
                if not _table in _tbls:
                    _tbls[_table] = _Table(_table, metadata, autoload_with = engine)
                for _,r in _df.iterrows():
                    #print(r['table'], list(tables))
                    if r['table'] in list(tables):
                        print(r['table'])
                        _data = dict(r)
                        _data['user_id'] = 1
                        _data['app_id'] = app.app_id
                        _data['created_at'] = now 
                        _data['updated_at'] = now 
                        _data['db'] = app_database
                        del _data[f'{_table}_id']
                        # UPSERT INTO THE NEW DB
                        sql = select(_tbls[_table].c)\
                                .select_from(_tbls[_table])\
                                .where(and_(
                                    _tbls[_table].c['table'] == r['table'],
                                    _tbls[_table].c.db == app_database,
                                    _tbls[_table].c.app_id == app.app_id
                                ))
                        _tdata = conn.execute(sql).fetchone()
                        if _tdata:
                            #_data[f'{_table}_id'] = _tdata[f'{_table}_id']
                            sql = update(_tbls[_table])\
                                .where(and_(
                                    _tbls[_table].c.db == app_database,
                                    _tbls[_table].c.table == _data.get('table'),
                                    #_tbls[_table].c[f'{_table}_id'] == _tdata[f'{_table}_id'],
                                )).values(_data)
                            result = conn.execute(sql)
                            result.close()
                        else:
                            sql = insert(_tbls[_table]).values(_data)
                            result = conn.execute(sql)
                            result.close()
        except Exception as _err:
            print(str(_err))
    conn.commit()
    conn.close()

etl_report_base
etl_rbase_input
etl_rbase_output
etl_rbase_quality
etl_rb_reconcilia
etl_rbase_export
etl_report_base_log
manage_query
etl_rb_output_field
etl_rb_reconc_dtail
etl_rb_exp_dtail
dashboard
etl_rbase_backup
etl_rbase_notify
etl_report_base
etl_rbase_input
etl_rb_output_field
etl_rbase_output
etl_rbase_quality
etl_rb_reconcilia
etl_rb_reconc_dtail
etl_rbase_export
etl_rb_exp_dtail
etl_rbase_notify
etl_rbase_backup
dashboard
